In [ ]:
import astropy.coordinates as coord
import astropy.units as u
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import rcParams
from matplotlib import rc
import matplotlib as mpl
from scipy.integrate import quad
from scipy.optimize import curve_fit
from pypopsyn.simulator.configuration import cfg
import pypopsyn.simulator.basics.constants as const
import pypopsyn.simulator.multiband_emission.emission_radio as er

import utilities.plot_settings

In [ ]:
data = pd.read_pickle("../toremove_1/final_population.pkl.gz", compression="gzip")
data.head()

In [ ]:
x = data["x"]["[kpc]"].to_numpy()
y = data["y"]["[kpc]"].to_numpy()
z = data["z"]["[kpc]"].to_numpy()
RA = data["RA"]["[deg]"].to_numpy()
DEC = data["DEC"]["[deg]"].to_numpy()
pm_RA = data["pm_RA"]["[mas yr^-1]"].to_numpy()
pm_DEC = data["pm_DEC"]["[mas yr^-1]"].to_numpy()
v_r = data["v_r"]["[km s^-1]"].to_numpy()
v_phi = data["v_phi"]["[km s^-1]"].to_numpy()
v_z = data["v_z"]["[km s^-1]"].to_numpy()
dist = data["d"]["[kpc]"].to_numpy()
B = data["B"]["[G]"].to_numpy()
chi = data["chi"]["[rad]"].to_numpy()
P = data["P"]["[s]"].to_numpy()
P_dot = data["P_dot"]["[s yr^-1]"].to_numpy()
L_radio = data["L_radio"]["[erg s^-1 Hz^-1]"].to_numpy()
S_radio_Jy = data["S_radio"]["[Jy]"].to_numpy()
w_int = data["w_int"]["[s]"].to_numpy()
intercepted_radio = data["intercepted_radio"][" "].to_numpy(dtype=bool)
detected_radio_PMPS = data["detected_radio_PMPS"][" "].to_numpy(dtype=bool)
detected_radio_SMPS = data["detected_radio_SMPS"][" "].to_numpy(dtype=bool)
age = data["age"]["[yr]"].to_numpy()

print(len(detected_radio_PMPS[detected_radio_PMPS==True])/len(detected_radio_PMPS))
print(len(detected_radio_SMPS[detected_radio_SMPS==True])/len(detected_radio_SMPS))

print(len(x[detected_radio_PMPS]))
print(len(x[detected_radio_SMPS]))
print(len(x[detected_radio_PMPS | detected_radio_SMPS]))

In [ ]:
fig, ax = plt.subplots()

ax.plot(
    x,
    y,
    linestyle="None",
    marker="o",
    color="darkgray",
    markersize=1,
    alpha=0.1,
    rasterized=True
)

ax.plot(
    x[detected_radio_PMPS],
    y[detected_radio_PMPS],
    linestyle="None",
    marker="o",
    color="tab:blue",
    markersize=5,
    alpha=1.,
    rasterized=True
)
ax.plot(
    x[detected_radio_SMPS],
    y[detected_radio_SMPS],
    linestyle="None",
    marker="x",
    color="tab:red",
    markersize=10,
    alpha=1.,
    rasterized=True
)

ax.plot(0.0, 8.3, marker="o", color="tab:orange", markersize=6)
ax.set_xlabel(r"$x$ [kpc]")
ax.set_ylabel(r"$y$ [kpc]")
ax.set_xlim(-20.0, 20.0)
ax.set_ylim(-20.0, 20.0)

plt.show()

In [ ]:
fig, ax = plt.subplots()

ax.plot(
    x,
    z,
    linestyle="None",
    marker="o",
    color="darkgray",
    markersize=1,
    alpha=0.2,
    rasterized=True,
)

ax.plot(
    x[detected_radio_PMPS],
    z[detected_radio_PMPS],
    linestyle="None",
    marker="o",
    color="tab:blue",
    markersize=5,
    alpha=1.,
    rasterized=True
)
ax.plot(
    x[detected_radio_SMPS],
    z[detected_radio_SMPS],
    linestyle="None",
    marker="x",
    color="tab:red",
    markersize=10,
    alpha=1.,
    rasterized=True
)

ax.plot(0.0, 0.02, marker="o", color="tab:orange", markersize=6)
ax.set_xlabel(r"$x$ [kpc]")
ax.set_ylabel(r"$z$ [kpc]")
ax.set_xlim(-20.0, 20.0)
ax.set_ylim(-5.0, 5.0)

plt.show()

In [ ]:
r = np.sqrt(x**2 + y**2)
r_bins = np.linspace(0.0, 30.0, 31)

fig, ax = plt.subplots(figsize=(15,8))

ax.hist(
    r,
    bins=r_bins,
    histtype="step",
    edgecolor="darkgray",
    lw=4,
    alpha=1,
    label="Simulated all"
)
ax.hist(
    r[detected_radio_PMPS],
    bins=r_bins,
    histtype="step",
    edgecolor="tab:blue",
    lw=4,
    alpha=1,
    label="detected by PMPS",
)
ax.hist(
    r[detected_radio_SMPS],
    bins=r_bins,
    histtype="step",
    edgecolor="tab:red",
    lw=4,
    alpha=1,
    label="detected by SMPS",
)
plt.xlabel(r"$r$ [kpc]")
plt.ylabel(r"Number of NSs")
plt.xlim(0.0, 30.0)
plt.yscale('log')
plt.legend(bbox_to_anchor=(1.05, 1), frameon=False, loc=0, fontsize=20)

plt.show()

In [ ]:
z_bins = np.linspace(0.0, 5.0, 31)

fig, ax = plt.subplots(figsize=(15,8))

ax.hist(
    z,
    bins=z_bins,
    histtype="step",
    edgecolor="darkgray",
    lw=4,
    alpha=1,
    label="Simulated all",
)
ax.hist(
    z[detected_radio_PMPS],
    bins=z_bins,
    histtype="step",
    edgecolor="tab:blue",
    lw=4,
    alpha=1,
    label="detected by PMPS",
)
ax.hist(
    z[detected_radio_SMPS],
    bins=z_bins,
    histtype="step",
    edgecolor="tab:red",
    lw=4,
    alpha=1,
    label="detected by SMPS",
)

plt.xlabel(r"$z$ [kpc]")
plt.ylabel(r"Number of NS")
plt.xlim(0.0, 5.0)
plt.ylim(0.1, 1.e5)
plt.yscale('log')
plt.legend(bbox_to_anchor=(1.05, 1), frameon=False, loc=0, fontsize=20)

plt.show()

In [ ]:
v_tot = np.sqrt(v_r**2 + v_phi**2 + v_z**2)

fig, ax = plt.subplots(figsize=(15,8))
v_bins = np.linspace(0,1500.,31)  

ax.hist(
    v_tot,
    bins=v_bins,
    histtype="step",
    edgecolor="darkgray",
    lw=4,
    alpha=1,
    label=r"Simulated all",
)
ax.hist(
    v_tot[detected_radio_PMPS],
    bins=v_bins,
    histtype="step",
    edgecolor="tab:blue",
    lw=4,
    alpha=1,
    label=r"detecteed by PMPS",
)
ax.hist(
    v_tot[detected_radio_SMPS],
    bins=v_bins,
    histtype="step",
    edgecolor="tab:red",
    lw=4,
    alpha=1,
    label=r"detecteed by SMPS",
)
ax.set_xlabel(r"Total velocity magnitude [km s$^{-1}$]")
ax.set_ylabel(r"Number of NSs")
plt.yscale('log')
plt.ylim(0.1, 3.e6)
plt.legend(bbox_to_anchor=(1.05, 1), frameon=False, loc=0, fontsize=20)

plt.show()

In [ ]:
P_bins = np.logspace(-2., 2., 31)

fig, ax = plt.subplots(figsize=(15,8))

ax.hist(
    P[detected_radio_PMPS],
    bins=P_bins,
    histtype="step",
    edgecolor="tab:blue",
    lw=4,
    label="detected by PMPS",
)
ax.hist(
    P[detected_radio_SMPS],
    bins=P_bins,
    histtype="step",
    edgecolor="tab:red",
    lw=4,
    label="detected by SMPS",
)
plt.xlabel(r"$P$ [s]")
plt.ylabel(r"Number of NSs")
#plt.xlim(0., 50.0)
plt.ylim(0.1, 1.e3)
plt.xscale('log')
plt.yscale('log')
plt.legend(bbox_to_anchor=(1.05, 1), frameon=False, loc=0, fontsize=20)

plt.show()

In [ ]:
B_log10_bins = np.linspace(9.0, 17.0, 31)

fig, ax = plt.subplots(figsize=(15,8))

ax.hist(
    np.log10(B[detected_radio_PMPS]),
    bins=B_log10_bins,
    histtype="step",
    edgecolor="tab:blue",
    lw=4,
    label="detected by PMPS",
)
ax.hist(
    np.log10(B[detected_radio_SMPS]),
    bins=B_log10_bins,
    histtype="step",
    edgecolor="tab:red",
    lw=4,
    label="detected by SMPS",
)
plt.xlabel(r"log$_{10} B$ [G]")
plt.ylabel(r"Number of NSs")
plt.xlim(9., 17.0)
plt.ylim(0.1, 5.e4)
plt.yscale('log')
plt.legend(bbox_to_anchor=(1.05, 1), frameon=False, loc=0, fontsize=20)

plt.show()

In [ ]:
chi_bins = np.linspace(0, np.pi / 2, 31)

fig, ax = plt.subplots(figsize=(15,8))

ax.hist(
    chi[detected_radio_PMPS],
    bins=chi_bins,
    histtype="step",
    edgecolor="tab:blue",
    lw=4,
    label="detected by PMPS",
)
ax.hist(
    chi[detected_radio_SMPS],
    bins=chi_bins,
    histtype="step",
    edgecolor="tab:red",
    lw=4,
    label="detected by SMPS",
)
plt.xlabel(r"$\chi$ [rad]")
plt.ylabel(r"Normalized PDF")
plt.xlim(0., np.pi / 2)
plt.ylim(0.1, 1.e3)
plt.yscale('log')
plt.legend(bbox_to_anchor=(1.05, 1), frameon=False, loc=0, fontsize=20)

plt.show()

In [ ]:
fig, ax = plt.subplots()

ax.plot(
    P[detected_radio_PMPS],
    P_dot[detected_radio_PMPS] / const.YR_TO_S,
    linestyle="None",
    marker="o",
    color="tab:blue",
    markersize=5,
    alpha=1.,
    rasterized=True
)
ax.plot(
    P[detected_radio_SMPS],
    P_dot[detected_radio_SMPS] / const.YR_TO_S,
    linestyle="None",
    marker="x",
    color="tab:red",
    markersize=10,
    alpha=1.,
    rasterized=True
)

ax.set_xscale('log')
ax.set_yscale('log')

plt.xlabel(r"$P$ [s]")
plt.ylabel(r"$\dot{P}$ [s/s]")

plt.show()

In [ ]:
S_radio_bins = np.logspace(-5, 1, 31)

fig, ax = plt.subplots(figsize=(15,8))

ax.hist(
    S_radio_Jy[detected_radio_PMPS],
    bins=S_radio_bins,
    histtype="step",
    edgecolor="tab:blue",
    lw=4,
    label="detected by PMPS",
)
ax.hist(
    S_radio_Jy[detected_radio_SMPS],
    bins=S_radio_bins,
    histtype="step",
    edgecolor="tab:red",
    lw=4,
    label="detected by SMPS",
)
plt.xlabel(r"$S_{\rm radio}$ [Jy]")
plt.ylabel(r"Number of NSs")
ax.set_xscale('log')
ax.set_yscale('log')
plt.legend(bbox_to_anchor=(1.05, 1), frameon=False, loc=0, fontsize=20)

plt.show()

In [ ]:
w_int_deg = w_int / P * 360.

fig, ax = plt.subplots(figsize=(15,8))

ax.loglog(P[detected_radio_PMPS], w_int_deg[detected_radio_PMPS], 'o', color='tab:blue', ms=6, rasterized=True,label="detected by PMPS",)
ax.loglog(P[detected_radio_SMPS], w_int_deg[detected_radio_SMPS], 'x', color='tab:red', ms=6, rasterized=True,label="detected by SMPS",)
plt.xlabel(r"$P$ [s]")
plt.ylabel(r"$w$ [deg]")

plt.legend(bbox_to_anchor=(1.05, 1), frameon=False, loc=0, fontsize=20)
plt.show(block=False)